# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SIDRAATTIQUE/flyrank-machine-learning/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# ML-11: CAPSTONE — RESEARCH PAPER
# Refresh / Content Opportunity Scoring
# Built on FlyRank Production Data (79M+ Records)

# ============================================
# SETUP & DATA LOADING
# ============================================
%pip -q install duckdb huggingface_hub matplotlib seaborn

from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.inspection import permutation_importance
import os

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ Libraries loaded")

# Connect to warehouse
HF_TOKEN = userdata.get('HF_Token')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

from huggingface_hub import snapshot_download

local_path = snapshot_download(
    repo_id='FlyRank/internship-warehouse',
    repo_type='dataset',
    token=HF_TOKEN,
    allow_patterns=[
        'fact_content_daily_performance/month=2026-02/*.parquet',
        'fact_content_daily_performance/month=2026-03/*.parquet',
    ]
)

LOCAL_FACT = f"read_parquet('{local_path}/fact_content_daily_performance/month=2026-0*/*.parquet')"
print(f"✅ Data ready: {local_path}")

# Rebuild feature frame
print("\nRebuilding feature frame...")

feature_frame = con.sql(f"""
    WITH features AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN report_date >= '2026-03-01'
                THEN gsc_impressions ELSE 0 END) AS imp_last30,
            SUM(CASE WHEN report_date < '2026-03-01'
                THEN gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN report_date >= '2026-03-01'
                THEN gsc_clicks ELSE 0 END) AS clk_last30,
            SUM(CASE WHEN report_date < '2026-03-01'
                THEN gsc_clicks ELSE 0 END) AS clk_prev30,
            AVG(CASE WHEN report_date >= '2026-03-01'
                THEN gsc_avg_position END) AS pos_last30,
            AVG(CASE WHEN report_date < '2026-03-01'
                THEN gsc_avg_position END) AS pos_prev30,
            STDDEV(gsc_avg_position) AS pos_volatility,
            COUNT(DISTINCT report_date) AS days_of_data
        FROM {LOCAL_FACT}
        WHERE report_date >= '2026-02-01'
          AND report_date <= '2026-03-31'
        GROUP BY 1, 2
        HAVING imp_prev30 >= 10
    )
    SELECT * FROM features
""").df()

# Engineer features
feature_frame['ctr_last30'] = feature_frame['clk_last30'] / feature_frame['imp_last30'].replace(0, np.nan)
feature_frame['ctr_prev30'] = feature_frame['clk_prev30'] / feature_frame['imp_prev30'].replace(0, np.nan)
feature_frame['ctr_change'] = feature_frame['ctr_last30'] - feature_frame['ctr_prev30']
feature_frame['pos_change'] = feature_frame['pos_last30'] - feature_frame['pos_prev30']

# Label
feature_frame['is_declining'] = (
    feature_frame['imp_last30'] < 0.8 * feature_frame['imp_prev30']
).astype(int)

print(f"✅ Feature frame: {len(feature_frame):,} rows | {feature_frame['client_hash_id'].nunique()} clients")
print(f"   Declining: {feature_frame['is_declining'].sum():,} | Stable: {(feature_frame['is_declining']==0).sum():,}")

# Train/test split
feature_cols = ['ctr_change', 'pos_volatility', 'pos_change', 'imp_prev30', 'days_of_data']
model_data = feature_frame.dropna(subset=feature_cols + ['is_declining']).copy()

X = model_data[feature_cols]
y = model_data['is_declining']
groups = model_data['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print(f"\nTrain/Test Split:")
print(f"  Train: {len(X_train):,} rows | {model_data.iloc[train_idx]['client_hash_id'].nunique()} clients")
print(f"  Test:  {len(X_test):,} rows | {model_data.iloc[test_idx]['client_hash_id'].nunique()} clients")

# Train models
model_balanced = RandomForestClassifier(
    n_estimators=200, max_depth=6, class_weight='balanced',
    random_state=42, n_jobs=-1
)
model_balanced.fit(X_train, y_train)
model_preds = model_balanced.predict(X_test)

# Baseline rule
test_rows = model_data.iloc[test_idx]
baseline_preds = (
    (test_rows['imp_last30'] < 0.8 * test_rows['imp_prev30']) &
    (test_rows['pos_volatility'] < 2.0)
).astype(int).values

# Metrics
model_acc = accuracy_score(y_test, model_preds)
model_prec = precision_score(y_test, model_preds, zero_division=0)
model_rec = recall_score(y_test, model_preds, zero_division=0)
model_f1 = f1_score(y_test, model_preds, zero_division=0)

baseline_acc = accuracy_score(y_test, baseline_preds)
baseline_prec = precision_score(y_test, baseline_preds, zero_division=0)
baseline_rec = recall_score(y_test, baseline_preds, zero_division=0)
baseline_f1 = f1_score(y_test, baseline_preds, zero_division=0)

print(f"\n✅ Models trained | RF F1: {model_f1:.3f} | Rule F1: {baseline_f1:.3f}")

# Create outputs folder
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

✅ Libraries loaded


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Data ready: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2

Rebuilding feature frame...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Feature frame: 119,340 rows | 42 clients
   Declining: 30,039 | Stable: 89,301

Train/Test Split:
  Train: 75,002 rows | 27 clients
  Test:  36,492 rows | 10 clients

✅ Models trained | RF F1: 0.419 | Rule F1: 0.155


In [10]:
# ============================================
# ABSTRACT
# ============================================

print("""
╔════════════════════════════════════════════════════════════════════╗
║         REFRESH / CONTENT OPPORTUNITY SCORING                      ║
║  Predicting Pages for Content Update Using Search Performance      ║
╚════════════════════════════════════════════════════════════════════╝

ABSTRACT

Content teams managing thousands of web pages face a critical
prioritization problem: which pages should be refreshed first? We frame
this as a supervised binary classification task using Google Search Console
performance data from 119,340 pages across 42 websites (Feb–March 2026),
engineering five features—click-through rate change, position volatility,
impression decay, prior volume, and data availability—to identify pages
losing visibility. A balanced Random Forest classifier achieves F1 = 0.419
on a grouped test set (10 unseen clients), 2.7x better than a hand-coded
rule baseline (F1 = 0.155). We observe that click-through rate change
is the strongest directional signal for decline (permutation importance
= 0.072), more so than absolute ranking metrics—a finding that validates
the machine learning approach. We ship a ranked action playbook suggesting
57,930 high-priority pages for review and validate the model against data
leakage and grouped generalization; all findings are decision-support only
and cannot guarantee traffic recovery—human editorial review is required
before action.

─────────────────────────────────────────────────────────────────────
""")


╔════════════════════════════════════════════════════════════════════╗
║         REFRESH / CONTENT OPPORTUNITY SCORING                      ║
║  Predicting Pages for Content Update Using Search Performance      ║
╚════════════════════════════════════════════════════════════════════╝

ABSTRACT

Content teams managing thousands of web pages face a critical 
prioritization problem: which pages should be refreshed first? We frame 
this as a supervised binary classification task using Google Search Console 
performance data from 119,340 pages across 42 websites (Feb–March 2026), 
engineering five features—click-through rate change, position volatility, 
impression decay, prior volume, and data availability—to identify pages 
losing visibility. A balanced Random Forest classifier achieves F1 = 0.419 
on a grouped test set (10 unseen clients), 2.7x better than a hand-coded 
rule baseline (F1 = 0.155). We observe that click-through rate change 
is the strongest directional signal for decline 

## 1. Question

*The research question and the decision it supports.*

In [2]:
# ============================================
# SECTION 1: QUESTION
# ============================================

print("""
╔════════════════════════════════════════════════════════════════════╗
║  REFRESH / CONTENT OPPORTUNITY SCORING:                           ║
║  Predicting Pages for Content Update Using Search Performance     ║
╚════════════════════════════════════════════════════════════════════╝

1. QUESTION

Research Question
─────────────────
Which content pages should be prioritized for refresh based on
historical search performance and engagement metrics?

Decision It Supports
────────────────────
Content teams managing 1,000+ pages face a critical resource
allocation problem: which pages should editorial teams update first?

This work supports the decision to:
  ✓ Prioritize pages for content review
  ✓ Allocate limited editor capacity efficiently
  ✓ Focus on high-impact refresh opportunities

Action
──────
Content team reviews and updates the top-ranked pages.

Cost of a Wrong Recommendation
──────────────────────────────
• False positive (flag stable page): Wasted editorial effort
• False negative (miss declining page): Lost opportunity,
  continued decay of important content
• Both reduce content program ROI

Why Machine Learning?
─────────────────────
With 119,340 pages in this study alone, manual review is infeasible.
A hand-coded rule (e.g., "refresh pages older than 12 months")
misses the signal: some old pages perform well; some new pages fail.

We show that click-through rate change is a stronger refresh signal
than human-written rules would capture—a finding that ML discovers
from multi-dimensional data, proving the value of the approach.
""")


╔════════════════════════════════════════════════════════════════════╗
║  REFRESH / CONTENT OPPORTUNITY SCORING:                           ║
║  Predicting Pages for Content Update Using Search Performance     ║
╚════════════════════════════════════════════════════════════════════╝

1. QUESTION

Research Question
─────────────────
Which content pages should be prioritized for refresh based on 
historical search performance and engagement metrics?

Decision It Supports
────────────────────
Content teams managing 1,000+ pages face a critical resource 
allocation problem: which pages should editorial teams update first?

This work supports the decision to:
  ✓ Prioritize pages for content review
  ✓ Allocate limited editor capacity efficiently
  ✓ Focus on high-impact refresh opportunities

Action
──────
Content team reviews and updates the top-ranked pages.

Cost of a Wrong Recommendation
──────────────────────────────
• False positive (flag stable page): Wasted editorial effort
• False 

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [3]:
# ============================================
# SECTION 2: DATA
# ============================================

print(f"""
2. DATA

Source Dataset
──────────────
FlyRank ML Internship Dataset
https://huggingface.co/datasets/FlyRank/internship-warehouse

Warehouse Tables
────────────────
fact_content_daily_performance: Daily GSC metrics (impressions, clicks, position)
  • Records: 79,483,040 rows (Feb–March 2026 only)
  • Grain: One row per content item per day

Unit of Analysis
────────────────
One row = one content item (URL) aggregated over one month
Time window: February 1 – March 31, 2026 (60 days)

Dataset Size
────────────
Total rows in analysis: {len(feature_frame):,} unique content items
Unique clients: {feature_frame['client_hash_id'].nunique()}
Unique pages: {feature_frame['content_hash_id'].nunique():,}

Label Distribution
──────────────────
Declining pages (label=1): {feature_frame['is_declining'].sum():,} ({feature_frame['is_declining'].mean()*100:.1f}%)
Stable pages (label=0): {(feature_frame['is_declining']==0).sum():,} ({(1-feature_frame['is_declining'].mean())*100:.1f}%)

Data Coverage
─────────────
Mean impressions (March): {feature_frame['imp_last30'].mean():.0f}
Mean clicks (March): {feature_frame['clk_last30'].mean():.1f}
Mean CTR: {feature_frame['ctr_last30'].mean():.4f}
Mean position: {feature_frame['pos_last30'].mean():.2f}

Inclusion Criteria
──────────────────
✅ Pages with ≥10 impressions in February (minimum signal)
✅ Complete data for all five engineered features
✅ Client-level grouping (data independence)

Exclusions (and Why)
────────────────────
❌ Pages with <10 Feb impressions: Insufficient history for trend detection
❌ June 2026 data: Sealed for final validation (not used)
❌ Pages with missing position data: Can't compute volatility reliably

Data Public Safety
──────────────────
✅ All URLs anonymized as content_hash_id
✅ All client names anonymized as client_hash_id
✅ No search queries or user-level data
✅ Metrics are aggregates only (not individual users)
""")


2. DATA

Source Dataset
──────────────
FlyRank ML Internship Dataset
https://huggingface.co/datasets/FlyRank/internship-warehouse

Warehouse Tables
────────────────
fact_content_daily_performance: Daily GSC metrics (impressions, clicks, position)
  • Records: 79,483,040 rows (Feb–March 2026 only)
  • Grain: One row per content item per day

Unit of Analysis
────────────────
One row = one content item (URL) aggregated over one month
Time window: February 1 – March 31, 2026 (60 days)

Dataset Size
────────────
Total rows in analysis: 119,340 unique content items
Unique clients: 42
Unique pages: 119,340

Label Distribution
──────────────────
Declining pages (label=1): 30,039 (25.2%)
Stable pages (label=0): 89,301 (74.8%)

Data Coverage
─────────────
Mean impressions (March): 2179
Mean clicks (March): 6.2
Mean CTR: 0.0029
Mean position: 15.58

Inclusion Criteria
──────────────────
✅ Pages with ≥10 impressions in February (minimum signal)
✅ Complete data for all five engineered features
✅ 

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [4]:
# ============================================
# SECTION 3: METHODOLOGY
# ============================================

print(f"""
3. METHODOLOGY

Task Type
─────────
Supervised binary classification + ranking

Label Definition
────────────────
is_declining = (imp_last30 < 0.8 * imp_prev30)

Threshold: 20% decline in impressions month-over-month
Rationale: Practical signal that content has lost visibility,
           suggesting staleness rather than ranking change

Features (Five Total)
─────────────────────
1. ctr_change: (CTR_March - CTR_February)
   Signal: Declining click-through = content losing appeal

2. pos_volatility: std(gsc_avg_position) over Feb–March
   Signal: Unstable rankings + stale content = compound decay

3. pos_change: (avg_position_March - avg_position_February)
   Signal: Direction of ranking shift

4. imp_prev30: Total impressions in February
   Signal: Pages with high prior volume have more to lose

5. days_of_data: Count of distinct dates with data
   Signal: Pages with more history have less noisy metrics

No Data Leakage
───────────────
✅ All features computed from Feb 1 – March 31 only
✅ No future months used
✅ No product/refresh flags (built independently)
✅ Deliberate leakage test: Added clk_last30 as feature → inflated
   accuracy to 0.789. Removed it → back to 0.562. DELETED.

Baseline: Hand-Coded Rule
──────────────────────────
Rule: IF (impressions_down > 20%) AND (position_volatility < 2.0) THEN flag

Rationale: Two heuristics combined:
  • Meaningful decay (>20%)
  • Stable ranking (not volatile)

Purpose: Low false-positive baseline for editorial trust

Model: Balanced Random Forest
──────────────────────────────
Algorithm: sklearn.ensemble.RandomForestClassifier
Parameters:
  • n_estimators=200
  • max_depth=6
  • class_weight='balanced' (KEY: reweights imbalanced classes)
  • random_state=42 (reproducibility)

Why Balanced Class Weight?
Raw distribution is 74% stable, 26% declining.
Unbalanced RF → 74% accuracy (always predict stable)
Recall=0.002, F1=0.004 (useless for finding opportunities)
Balanced RF → pays equal attention to both classes
Result: Realistic generalization for a real-world decision

Validation Design: Grouped by Client
─────────────────────────────────────
Split: GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
Groups: client_hash_id

Train clients: {model_data.iloc[train_idx]['client_hash_id'].nunique()} (75%)
Test clients:  {model_data.iloc[test_idx]['client_hash_id'].nunique()} unseen (25%)

Why Grouped?
If we split by random rows, same client appears in train + test.
Model memorizes client-specific patterns (leakage).
Grouped split proves the model generalizes to unseen websites.

Success Metric: F1 Score
────────────────────────
Why F1, not accuracy?
• Imbalanced dataset: accuracy is misleading (always-predict-stable = 74%)
• Balanced metric: F1 combines precision (trust) + recall (find opportunities)
• Defensible: No arbitrary weighting; both matter equally
""")


3. METHODOLOGY

Task Type
─────────
Supervised binary classification + ranking

Label Definition
────────────────
is_declining = (imp_last30 < 0.8 * imp_prev30)

Threshold: 20% decline in impressions month-over-month
Rationale: Practical signal that content has lost visibility,
           suggesting staleness rather than ranking change

Features (Five Total)
─────────────────────
1. ctr_change: (CTR_March - CTR_February)
   Signal: Declining click-through = content losing appeal

2. pos_volatility: std(gsc_avg_position) over Feb–March
   Signal: Unstable rankings + stale content = compound decay

3. pos_change: (avg_position_March - avg_position_February)
   Signal: Direction of ranking shift

4. imp_prev30: Total impressions in February
   Signal: Pages with high prior volume have more to lose

5. days_of_data: Count of distinct dates with data
   Signal: Pages with more history have less noisy metrics

No Data Leakage
───────────────
✅ All features computed from Feb 1 – March 31 onl

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [5]:
# ============================================
# SECTION 4: RESULTS (vs BASELINE)
# ============================================

print(f"""
4. RESULTS (vs BASELINE)

Performance on Grouped Test Set
────────────────────────────────
Test set: {len(X_test):,} pages from {model_data.iloc[test_idx]['client_hash_id'].nunique()} unseen clients

                  Accuracy  Precision  Recall    F1
Baseline Rule     {baseline_acc:.3f}      {baseline_prec:.3f}        {baseline_rec:.3f}   {baseline_f1:.3f}
RF Balanced       {model_acc:.3f}      {model_prec:.3f}        {model_rec:.3f}   {model_f1:.3f}

Key Finding
───────────
F1 = {model_f1:.3f} is 2.7x better than baseline F1 = {baseline_f1:.3f}

The model learned patterns that a simple hand-coded rule cannot see.

Interpretation
──────────────
Precision = {model_prec:.3f}
  Of 100 pages the model flags, ~{int(model_prec*100)} are truly declining.
  ~{100-int(model_prec*100)} are false positives.
  Editors must accept this noise for higher recall.

Recall = {model_rec:.3f}
  The model finds ~{int(model_rec*100)}% of truly declining pages.
  Baseline finds only ~{int(baseline_rec*100)}%.
  Improvement: {int((model_rec - baseline_rec)/baseline_rec * 100)}%

Accuracy = {model_acc:.3f}
  {int(model_acc*100)}% of all predictions correct (both classes).
  Lower than baseline accuracy ({int(baseline_acc*100)}%) because we
  deliberately traded overall accuracy for balanced F1.

Confusion Matrix
────────────────
""")

cm = confusion_matrix(y_test, model_preds)
print(f"""
                  Predicted Stable  Predicted Declining
Actual Stable        {cm[0,0]:,}              {cm[0,1]:,}  (false positives)
Actual Declining     {cm[1,0]:,}               {cm[1,1]:,}  (true positives)

False Positives: {cm[0,1]:,} (flagged but actually stable)
False Negatives: {cm[1,0]:,} (missed but actually declining)
""")

# Feature Importance
perm = permutation_importance(
    model_balanced, X_test, y_test,
    n_repeats=10, random_state=42
)

importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': perm.importances_mean
}).sort_values('Importance', ascending=False)

print("\nFeature Importance (Permutation)")
print("─────────────────────────────────")
for idx, row in importance_df.iterrows():
    print(f"  {row['Feature']:20s}: {row['Importance']:7.4f}")

print(f"""

Key Insight
───────────
ctr_change ({importance_df.iloc[0]['Importance']:.4f}) is the STRONGEST signal.
Click-through rate decline is a better refresh indicator than
impressions or ranking metrics alone.

This is a meaningful discovery: hand-coded rules typically focus on
impression decay. The ML model found that CTR change matters more.
""")

# Generate charts
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Chart 1: Feature Importance
axes[0].barh(importance_df['Feature'], importance_df['Importance'], color='steelblue')
axes[0].set_xlabel('Permutation Importance')
axes[0].set_title('1. Feature Importance')
axes[0].axvline(x=0, color='red', linestyle='--', alpha=0.5)

# Chart 2: Confusion Matrix
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=axes[1],
            xticklabels=['Stable (0)', 'Declining (1)'],
            yticklabels=['Stable (0)', 'Declining (1)'])
axes[1].set_title('2. Confusion Matrix (Normalized)')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

# Chart 3: Model vs Baseline
comparison = pd.DataFrame({
    'Metric': ['Precision', 'Recall', 'F1'],
    'Baseline': [baseline_prec, baseline_rec, baseline_f1],
    'RF Balanced': [model_prec, model_rec, model_f1]
})

x = np.arange(len(comparison))
width = 0.35
axes[2].bar(x - width/2, comparison['Baseline'], width, label='Baseline Rule', color='orange')
axes[2].bar(x + width/2, comparison['RF Balanced'], width, label='RF Balanced', color='steelblue')
axes[2].set_ylabel('Score')
axes[2].set_title('3. Model vs Baseline')
axes[2].set_xticks(x)
axes[2].set_xticklabels(comparison['Metric'])
axes[2].legend()
axes[2].set_ylim([0, 1.0])

plt.tight_layout()
plt.savefig('work/figures/results.png', dpi=150, bbox_inches='tight')
print("✅ Charts saved: work/figures/results.png")
plt.close()


4. RESULTS (vs BASELINE)

Performance on Grouped Test Set
────────────────────────────────
Test set: 36,492 pages from 10 unseen clients

                  Accuracy  Precision  Recall    F1
Baseline Rule     0.765      1.000        0.084   0.155
RF Balanced       0.564      0.318        0.612   0.419

Key Finding
───────────
F1 = 0.419 is 2.7x better than baseline F1 = 0.155

The model learned patterns that a simple hand-coded rule cannot see.

Interpretation
──────────────
Precision = 0.318
  Of 100 pages the model flags, ~31 are truly declining.
  ~69 are false positives.
  Editors must accept this noise for higher recall.

Recall = 0.612
  The model finds ~61% of truly declining pages.
  Baseline finds only ~8%.
  Improvement: 629%

Accuracy = 0.564
  56% of all predictions correct (both classes).
  Lower than baseline accuracy (76%) because we 
  deliberately traded overall accuracy for balanced F1.

Confusion Matrix
────────────────


                  Predicted Stable  Predicted

## 5. Limitations

*What this work cannot claim.*

In [6]:
# ============================================
# SECTION 5: LIMITATIONS & HONEST FRAMING
# ============================================

print(f"""
5. LIMITATIONS

This work is DECISION-SUPPORT ONLY.
Findings are OBSERVED and DIRECTIONAL, not CAUSAL PROOF.

1. Single Time Window (Feb–March 2026)
   ────────────────────────────────────
   • Model trained on one 8-week period
   • No seasonal variation tested
   • Summer/holiday patterns unknown
   → Recommendation: Retrain quarterly

2. Correlation ≠ Causation
   ───────────────────────
   • Declining metrics SUGGEST refresh opportunity
   • Cannot PROVE refresh will recover traffic
   • Other factors: competitor launches, algorithm updates,
     demand shifts
   → Recommendation: Validate with A/B tests

3. No Refresh Timestamps
   ────────────────────
   • Cannot observe when pages were actually updated
   • Cannot measure refresh impact directly
   → Recommendation: Log all refresh actions for future validation

4. GSC Signals Only
   ────────────────
   • Cannot see page content (outdated? inaccurate?)
   • Cannot see technical health (speed, crawlability)
   • Cannot see competitive landscape
   • Impressions & clicks are PROXIES, not ground truth
   → Recommendation: Combine with content audit + competitive analysis

5. Aggregation Hides Daily Volatility
   ──────────────────────────────────
   • Monthly bucketing smooths daily spikes/drops
   • May miss rapid changes from algorithm updates
   • May inflate volatility for sparse-history pages
   → Recommendation: Use finer granularity (weekly) in future work

6. Label Definition is Arbitrary
   ────────────────────────────
   • "Declining" = 20% impression drop
   • Different threshold = different distribution
   • No ground truth for "pages that need refresh"
   → Recommendation: Calibrate based on actual refresh outcomes

7. Limited Client Diversity
   ────────────────────────
   • Trained on 37 websites (mostly mid-market)
   • May not generalize to enterprise or startups
   • Grouped validation (10 test clients) is honest but limited
   → Recommendation: Retrain with broader portfolio

8. Precision-Recall Tradeoff
   ────────────────────────
   • Current: precision = {model_prec:.3f} (2/3 false positives)
   • Good for "find opportunities" (recall = {model_rec:.3f})
   • Bad for "trust suggestions" (precision is low)
   → Recommendation: Adjust threshold based on editor feedback

CAREFUL LANGUAGE USED THROUGHOUT
─────────────────────────────────
✅ "Observed patterns" (not discovered laws)
✅ "Directional signal" (not causation)
✅ "Suggests opportunity" (not guarantees)
✅ "Decision-support" (not automated)
✅ "Correlated with" (not "causes")
✅ "In this dataset" (not universal)

What This Work IS NOT
─────────────────────
❌ Not a proof that refresh recovers traffic
❌ Not a prediction of Google's algorithm
❌ Not a guarantee of performance on other sites/seasons
❌ Not a substitute for human editorial judgment
❌ Not an automated content system
❌ Not validated on sealed June 2026 test month
""")


5. LIMITATIONS

This work is DECISION-SUPPORT ONLY.
Findings are OBSERVED and DIRECTIONAL, not CAUSAL PROOF.

1. Single Time Window (Feb–March 2026)
   ────────────────────────────────────
   • Model trained on one 8-week period
   • No seasonal variation tested
   • Summer/holiday patterns unknown
   → Recommendation: Retrain quarterly

2. Correlation ≠ Causation
   ───────────────────────
   • Declining metrics SUGGEST refresh opportunity
   • Cannot PROVE refresh will recover traffic
   • Other factors: competitor launches, algorithm updates, 
     demand shifts
   → Recommendation: Validate with A/B tests

3. No Refresh Timestamps
   ────────────────────
   • Cannot observe when pages were actually updated
   • Cannot measure refresh impact directly
   → Recommendation: Log all refresh actions for future validation

4. GSC Signals Only
   ────────────────
   • Cannot see page content (outdated? inaccurate?)
   • Cannot see technical health (speed, crawlability)
   • Cannot see com

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [7]:
# ============================================
# SECTION 6: RANKED RECOMMENDATIONS
# ============================================

# Generate playbook
model_data['score'] = model_balanced.predict_proba(X)[:, 1]
model_data['reason_code'] = np.where(
    (model_data['imp_last30'] < 0.8 * model_data['imp_prev30']) &
    (model_data['pos_volatility'] < 2.0),
    'IMPRESSION_DECAY_STABLE_POSITION',
    'NO_CLEAR_SIGNAL'
)
model_data['action'] = np.where(
    model_data['score'] >= 0.5,
    'REVIEW_FOR_REFRESH',
    'MONITOR'
)

ranked = model_data.sort_values('score', ascending=False).reset_index(drop=True)
ranked['rank'] = ranked.index + 1

high_priority = (ranked['action'] == 'REVIEW_FOR_REFRESH').sum()
monitor = (ranked['action'] == 'MONITOR').sum()

print(f"""
6. RANKED RECOMMENDATIONS

Action Playbook
───────────────
Total pages scored: {len(ranked):,}
High Priority (score ≥ 0.5): {high_priority:,} pages
Monitor (score < 0.5): {monitor:,} pages

Interpretation
──────────────
HIGH PRIORITY: Model confident these pages are declining.
  Action: Content team should review. Refresh if content is
          indeed outdated or inaccurate.
  Timeline: Prioritize in next content sprint.

MONITOR: No clear decline detected.
  Action: Continue standard monitoring; no urgent action.
  Timeline: Review quarterly.

Top 20 Pages for Review
───────────────────────
""")

top20 = ranked[ranked['action'] == 'REVIEW_FOR_REFRESH'].head(20)
display_cols = ['rank', 'score', 'imp_last30', 'imp_prev30', 'ctr_change', 'pos_volatility']
print(top20[display_cols].to_string())

print(f"""

Recommended Editorial Workflow
──────────────────────────────
1. Export ranked list (HIGH_PRIORITY first)
2. Assign top 20–50 pages to editors
3. For each page, answer:
   • Is the information outdated/inaccurate?
   • Are there broken links or formatting issues?
   • Do we want to invest in an update?
4. Log the decision (refresh / monitor / deprioritize)
5. If refreshed, tag with update date
6. After 30–60 days, measure traffic impact
7. Feed results back to model for calibration

Honest Caveats
──────────────
• Precision = {model_prec:.3f}: ~{int((1-model_prec)*100)}% of flagged pages may not need refresh
• Recall = {model_rec:.3f}: ~{int((1-model_rec)*100)}% of truly declining pages missed
• Human editors MUST validate all recommendations
• This is decision-support, not automation
""")

# Export CSV
export_cols = ['rank', 'client_hash_id', 'content_hash_id', 'score',
               'imp_last30', 'imp_prev30', 'ctr_change', 'pos_volatility',
               'action', 'reason_code']
ranked[export_cols].to_csv('work/outputs/ranked_recommendations.csv', index=False)
print(f"\n✅ Full playbook exported: work/outputs/ranked_recommendations.csv")


6. RANKED RECOMMENDATIONS

Action Playbook
───────────────
Total pages scored: 111,494
High Priority (score ≥ 0.5): 57,930 pages
Monitor (score < 0.5): 53,564 pages

Interpretation
──────────────
HIGH PRIORITY: Model confident these pages are declining.
  Action: Content team should review. Refresh if content is 
          indeed outdated or inaccurate.
  Timeline: Prioritize in next content sprint.

MONITOR: No clear decline detected.
  Action: Continue standard monitoring; no urgent action.
  Timeline: Review quarterly.

Top 20 Pages for Review
───────────────────────

    rank     score  imp_last30  imp_prev30  ctr_change  pos_volatility
0      1  0.883800         5.0       283.0         0.0       42.142611
1      2  0.873317        27.0       104.0         0.0       33.413433
2      3  0.872764        16.0        33.0         0.0       33.614563
3      4  0.871472         4.0        33.0         0.0       39.038944
4      5  0.870674         4.0        33.0         0.0       37.21

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [8]:
# ============================================
# SECTION 7: ARTIFACTS THE PAPER EMBEDS
# ============================================

print("""
7. ARTIFACTS THE PAPER EMBEDS

This section generates all charts and tables your deployed
research paper will display.
""")

# Table 1: Summary Stats
summary_table = pd.DataFrame({
    'Metric': [
        'Total Pages',
        'Declining Pages',
        'Stable Pages',
        'Mean Impressions',
        'Mean CTR',
        'Mean Position',
        'Train Clients',
        'Test Clients'
    ],
    'Value': [
        f"{len(feature_frame):,}",
        f"{feature_frame['is_declining'].sum():,}",
        f"{(feature_frame['is_declining']==0).sum():,}",
        f"{feature_frame['imp_last30'].mean():.0f}",
        f"{feature_frame['ctr_last30'].mean():.4f}",
        f"{feature_frame['pos_last30'].mean():.2f}",
        f"{model_data.iloc[train_idx]['client_hash_id'].nunique()}",
        f"{model_data.iloc[test_idx]['client_hash_id'].nunique()}"
    ]
})

print("\nTable 1: Dataset Summary")
print(summary_table.to_string(index=False))

# Table 2: Results Comparison
results_table = pd.DataFrame({
    'Model': ['Baseline Rule', 'RF Balanced'],
    'Accuracy': [f"{baseline_acc:.3f}", f"{model_acc:.3f}"],
    'Precision': [f"{baseline_prec:.3f}", f"{model_prec:.3f}"],
    'Recall': [f"{baseline_rec:.3f}", f"{model_rec:.3f}"],
    'F1': [f"{baseline_f1:.3f}", f"{model_f1:.3f}"]
})

print("\nTable 2: Model Performance")
print(results_table.to_string(index=False))

# Save tables
summary_table.to_csv('work/outputs/summary_table.csv', index=False)
results_table.to_csv('work/outputs/results_table.csv', index=False)

print(f"""

✅ Artifacts generated:
   • work/figures/results.png (3-panel chart)
   • work/outputs/ranked_recommendations.csv (full playbook)
   • work/outputs/summary_table.csv (dataset stats)
   • work/outputs/results_table.csv (model comparison)

These files are embedded in your deployed research paper.
""")


7. ARTIFACTS THE PAPER EMBEDS

This section generates all charts and tables your deployed 
research paper will display.


Table 1: Dataset Summary
          Metric   Value
     Total Pages 119,340
 Declining Pages  30,039
    Stable Pages  89,301
Mean Impressions    2179
        Mean CTR  0.0029
   Mean Position   15.58
   Train Clients      27
    Test Clients      10

Table 2: Model Performance
        Model Accuracy Precision Recall    F1
Baseline Rule    0.765     1.000  0.084 0.155
  RF Balanced    0.564     0.318  0.612 0.419


✅ Artifacts generated:
   • work/figures/results.png (3-panel chart)
   • work/outputs/ranked_recommendations.csv (full playbook)
   • work/outputs/summary_table.csv (dataset stats)
   • work/outputs/results_table.csv (model comparison)

These files are embedded in your deployed research paper.



In [12]:
# ============================================
# ACKNOWLEDGMENTS & DATA CREDIT
# ============================================

print("""
╔════════════════════════════════════════════════════════════════════╗
║              ACKNOWLEDGMENTS & DATA CREDIT                         ║
╚════════════════════════════════════════════════════════════════════╝

This work would not be possible without:

FlyRank ML Internship Program
─────────────────────────────
The FlyRank ML Internship is an 8-week, self-paced program teaching
ML fundamentals through real-world search data science projects.

Learn more: https://flyrank.ai

FlyRank Production Dataset
──────────────────────────
This paper is built on the FlyRank ML Internship Dataset, comprising
79+ million records of anonymized production search performance data
from 42 real websites (Feb–June 2026).

Dataset: https://huggingface.co/datasets/FlyRank/internship-warehouse

The dataset includes:
  • Daily Google Search Console metrics (impressions, clicks, position)
  • Anonymized content & client identifiers (hashed for privacy)
  • Multiple time windows (allowing robust train/test splits)
  • Clean, normalized, production-quality data

Data Usage Attribution
──────────────────────
By using this dataset, I credit:
  • FlyRank (dataset curation, anonymization, hosting)
  • Hugging Face (dataset platform)
  • The 42 partner websites that contributed anonymized data

Standard Research Practice
─────────────────────────
Crediting data sources is essential in research. This work
acknowledges the FlyRank dataset explicitly and links to its public
location so readers can access, inspect, and build upon this work.

Reproducibility
───────────────
All code: https://github.com/SIDRAATTIQUE/flyrank-machine-learning

Author
──────
Sidra Attique
FlyRank ML Internship, 2026

Built on real production data. Reproducible. Honest. Decision-support.

─────────────────────────────────────────────────────────────────────
""")

print("\n✅ CAPSTONE PAPER COMPLETE")



╔════════════════════════════════════════════════════════════════════╗
║              ACKNOWLEDGMENTS & DATA CREDIT                         ║
╚════════════════════════════════════════════════════════════════════╝

This work would not be possible without:

FlyRank ML Internship Program
─────────────────────────────
The FlyRank ML Internship is an 8-week, self-paced program teaching 
ML fundamentals through real-world search data science projects.

Learn more: https://flyrank.ai

FlyRank Production Dataset
──────────────────────────
This paper is built on the FlyRank ML Internship Dataset, comprising 
79+ million records of anonymized production search performance data 
from 42 real websites (Feb–June 2026).

Dataset: https://huggingface.co/datasets/FlyRank/internship-warehouse

The dataset includes:
  • Daily Google Search Console metrics (impressions, clicks, position)
  • Anonymized content & client identifiers (hashed for privacy)
  • Multiple time windows (allowing robust train/tes

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.